# 🏠 House Price Prediction with Neural Networks 

This notebook demonstrates the **fundamental workflow** to build, train, and evaluate a neural network in **PyTorch** for a regression task: predicting house prices.

## What we'll learn:
- **Data loading and preprocessing** from Kaggle
- **Exploratory data analysis** to understand the features
- **Building a neural network** with PyTorch (BatchNorm, Dropout)
- **Training with early stopping** to prevent overfitting
- **Evaluating model performance** on unseen test data
- **Visualizing results** and understanding predictions

---

## 📦 Step 0: Install Required Packages

Before we begin, we need to install the tools (libraries) that our code depends on:

| Library | Purpose |
|---------|----------|
| `torch` | PyTorch — the deep learning framework we use to build and train the neural network |
| `scikit-learn` | Provides utilities for data splitting and scaling |
| `kagglehub` | Downloads datasets directly from Kaggle |
| `pandas` | Works with tabular data (like an Excel spreadsheet) |
| `numpy` | Fast numerical operations on arrays |
| `matplotlib` / `seaborn` | Creating static charts and visualizations |

**Run this cell once** — it may take a minute to install everything.

In [ ]:
# ============================================
# INSTALLATION — Run this first!
# ============================================

%pip install torch scikit-learn kagglehub pandas numpy matplotlib seaborn --quiet

print("✅ All packages installed successfully!")

## 📦 Step 1: Import Libraries

We keep all our helper code in a file called `utils.py`.  
Running the cell below imports everything we need — all the libraries **and** the helper functions that handle the boilerplate parts of this notebook.

Think of it as: *installing = buying the toolbox, `from utils import *` = opening the whole toolbox on your workbench at once.*

> 💡 You can open `utils.py` at any time to see exactly what each helper function does.

In [ ]:
# Import all libraries and helper functions from our utils file
from utils import *

print("✅ Everything imported successfully!")

---

# 📊 1. Data Loading & Exploration

## About the Dataset

We use a **synthetic (computer-generated) house price dataset** from Kaggle containing **10,000 houses**.

Each house is described by these features:

| Feature | Description |
|---------|------------|
| `square_feet` | Size of the house in square feet |
| `num_rooms` | Number of rooms |
| `age` | Age of the house in years |
| `distance_to_city(km)` | Distance from city center in km |
| `price` | The house price — **this is what we want to predict!** |

The code below **automatically downloads** the dataset from Kaggle using `kagglehub`.

In [ ]:
# Download, load, and clean the dataset — all in one step!
df = load_dataset()

### Visual exploration

Before building a model, it is always a good idea to **visualize** the data. We create four plots:

1. **Price distribution** — is it symmetric? skewed? are there outliers?
2. **Feature correlations** — which features are most/least related to price?
3. **Size vs Price** scatter — the single strongest predictor.
4. **Age vs Price** scatter — does age make a house cheaper or more expensive?

In [ ]:
# Show 4 charts to explore the data before building the model
plot_dataset(df)

---

# 🔧 2. Data Preparation

Raw data cannot be fed directly into a neural network — we need preparation steps:

1. **Separate features and target** — the model receives the features as input and tries to predict the target (price).
2. **Split** the data into training, validation, and test sets.
3. **Scale** (standardize) the values so all features live on a similar numeric range.
4. **Convert** to PyTorch tensors.

## Why three sets?

| Set | % of data | Role |
|-----|-----------|------|
| Training | 80% | The model **learns** from this data |
| Validation | 10% | Used to **monitor** training and detect overfitting |
| Test | 10% | **Final evaluation** on data the model has never seen |

### 2. Data Preparation

The `prepare_data(df)` function takes care of all four steps at once:

- **2a.** Separates the features (square_feet, num_rooms, age, distance_to_city) from the target (price)  
- **2b.** Splits the data into 80% training / 10% validation / 10% test  
- **2c.** Standardizes all values so every feature has mean = 0 and std = 1  
- **2d.** Converts everything to PyTorch tensors  

It returns the tensors we need for training, the original test prices in dollars, and the price scaler (needed to convert predictions back to dollars).

> 💡 Open `utils.py` to see the full implementation of `prepare_data()`.

In [ ]:
# Prepare the data: split into train/val/test, scale, and convert to tensors
(train_features_t, #
 val_features_t,
 test_features_t,
 train_prices_t,
 val_prices_t,
 test_prices_t,
 test_prices_original,
 price_scaler) = prepare_data(df)

---

# 🧠 3. Neural Network Architecture

## What is a neural network?

A neural network is a series of **layers** of interconnected "neurons". Each layer takes numbers in, applies a weighted sum plus a non-linear function, and passes the result to the next layer. By adjusting the weights during training, the network learns to map inputs to the desired output.

## Our architecture

We use a **4-layer fully-connected network**:

```
Input (4 features) → 128 → 128 → 64 → 1 (scaled price)
```

```
  ┌───────────┐     ┌───────────┐     ┌───────────┐     ┌───────────┐     ┌──────────┐
  │  INPUT    │     │  LAYER 1  │     │  LAYER 2  │     │  LAYER 3  │     │  OUTPUT  │
  │ (4 feat.) │────▶│128 neurons│────▶│128 neurons│────▶│ 64 neurons│────▶│ 1 neuron │
  │           │     │           │     │           │     │           │     │          │
  │sq_feet    │     │ Linear    │     │ Linear    │     │ Linear    │     │ Linear   │
  │num_rooms  │     │ ReLU      │     │ ReLU      │     │ ReLU      │     │ (no act.)│
  │age        │     │           │     │           │     │           │     │          │
  │distance   │     │           │     │           │     │           │     │          │
  └───────────┘     └───────────┘     └───────────┘     └───────────┘     └──────────┘
```

### Key components

| Component | What it does |
|-----------|-------------|
| `nn.Linear(in, out)` | A fully-connected layer: computes `out = weight × in + bias` |
| `nn.ReLU()` | Activation function: keeps positive values, sets negatives to 0 |

---

### 🎯 Task 1 — Add the second hidden layer

Layer 1 (128 neurons) is already defined in the code cell below. Your task is to add **Layer 2** immediately after it, following the same 2-component pattern.

> 💡 Hint: look at Layer 1 directly above the marked section — Layer 2 has the same structure and the same width.

In [ ]:
# ============================================
# DEFINE THE NEURAL NETWORK MODEL
# ============================================

class HousePriceModel(nn.Module):
    """
    A neural network for house price prediction.

    Architecture:
        Input → 128 → 128 → 64 → 1
        with ReLU activations.
    """

    def __init__(self, n_features):
        super().__init__()

        self.network = nn.Sequential(
            # Layer 1: input features → 128 neurons
            nn.Linear(n_features, 128),
            nn.ReLU(),

            # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            # 🎯 YOUR CODE HERE — Layer 2: 128 → 128 neurons
            # HINT: use the same 2-component pattern as Layer 1 above.
            # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            nn.Linear(128, 128),
            nn.ReLU(),

            # Layer 3: 128 → 64 neurons
            nn.Linear(128, 64),
            nn.ReLU(),

            # Output layer: 64 → 1 (predicted scaled price)
            nn.Linear(64, 1)
        )

    def forward(self, x):
        """Forward pass: push input x through all layers."""
        return self.network(x)


# Instantiate the model
model = HousePriceModel(
    n_features=train_features_t.shape[1]  # 4 features
)

print(model)
print(f"\n📊 Total parameters: {sum(p.numel() for p in model.parameters()):,}")

---

# 🏋️ 4. Training the Model

Training means repeatedly:

1. **Forward pass** — the model makes predictions for all training houses.
2. **Loss computation** — we measure how wrong the predictions are (Mean Squared Error).
3. **Backward pass** — PyTorch calculates the gradient of the loss with respect to every weight.
4. **Optimizer step** — the optimizer adjusts the weights to reduce the loss.

One full cycle through the training set is called an **epoch**.

## Hyperparameters

| Hyperparameter | Value | Meaning |
|---------------|-------|---------|
| `learning_rate` | 0.001 | How big each weight update step is |
| `weight_decay` | 1e-4 | L2 regularization — penalizes large weights |
| `max_epochs` | 2000 | Maximum training loops (early stopping will likely stop earlier) |

## Training strategy

In this base approach we use **full-batch training**: at each epoch, the model sees the **entire** training set at once. This is the simplest approach — we'll see an alternative (mini-batches) in the next notebook.

We also use **early stopping**: if the validation loss doesn't improve for 100 consecutive epochs, we stop training and restore the best model weights.

---

### 🎯 Task 2 — Set up the loss function and optimizer

In the next cell, fill in the five missing values.

> 💡 Hints:
> - The loss function measures the average squared difference between predictions and targets — `nn` has a built-in class for this.
> - All hyperparameter values are in the table above.
> - The optimizer is a popular adaptive variant of gradient descent; it needs the model's parameters and the learning rate.

In [ ]:
# ============================================
# SET UP LOSS FUNCTION AND OPTIMIZER
# ============================================

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 🎯 YOUR CODE HERE — fill in the five missing values

# MSELoss = Mean Squared Error: average of (prediction - actual)²
loss_function = nn.MSELoss()

# Learning rate — controls step size during optimization
learning_rate = 0.001

# Weight decay — L2 regularization strength
weight_decay = 1e-4

# Adam optimizer — a popular gradient-descent variant that adapts
# the learning rate for each parameter individually.
# HINT: Use model.parameters() to pass the model's parameters to the optimizer.
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

number_of_epochs = 2000

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print(f"Loss function: {loss_function}")
print(f"Optimizer:     Adam (lr={learning_rate}, weight_decay={weight_decay})")
print(f"Max epochs:    {number_of_epochs}")

### The training loop

Each epoch, `run_epoch()` handles the boilerplate for us:

1. **Train** — runs a forward pass on the full training set, computes the loss, back-propagates gradients, and updates weights.
2. **Validate** — computes the validation loss without updating weights, so we can monitor generalization.

Both steps return **float** loss values that we append to `train_losses` and `val_losses`.

We also implement **early stopping**: we track the best validation loss and if it doesn't improve for `patience` epochs, we stop and restore the best weights.

---

### 🎯 Task 3 — Implement early stopping

The training and validation phases are handled by `run_epoch()`. Your task is to fill in the **three missing pieces** inside the early stopping block.

> 💡 Hints:
> - When the model improves: remember the new best loss and start the patience count from scratch.
> - When it doesn't improve: track how many consecutive epochs have passed without improvement.
> - When patience runs out: stop the training loop and inform the user.

In [ ]:
# ============================================
# TRAINING LOOP (with Early Stopping)
# ============================================

# Lists to store loss values for later plotting
train_losses = []
val_losses   = []

# Early stopping setup
best_val_loss = float('inf')
patience_counter = 0
patience = 100  # Stop if no improvement for 100 epochs
best_model_state = None

for epoch in range(number_of_epochs):

    # Train for one epoch and evaluate on the validation set
    train_loss, val_loss = run_epoch(
        model, optimizer, loss_function,
        train_features_t, train_prices_t,
        val_features_t, val_prices_t
    )

    # Store losses
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # --- EARLY STOPPING CHECK ---
    if val_loss < best_val_loss:
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # 🎯 YOUR CODE HERE — improvement detected
        # HINT: store the new best loss and reset the stagnation counter
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        best_val_loss = val_loss
        patience_counter = 0

        # Save the best model state  (already done for you)
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # 🎯 YOUR CODE HERE — no improvement
        # HINT: increment the stagnation counter by 1
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        patience_counter += 1

    if patience_counter >= patience:
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # 🎯 YOUR CODE HERE — patience exhausted
        # HINT: exit the training loop and inform the user
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        print(f"\n⏹️  Early stopping triggered at epoch {epoch}")
        print(f"   No improvement for {patience} epochs")
        break

    # Print progress every 100 epochs
    if epoch % 100 == 0:
        print(
            f"Epoch {epoch:>4d} | "
            f"Train: {train_loss:.4f} | "
            f"Val: {val_loss:.4f}"
        )

# Load the best model state
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\n✅ Loaded best model from epoch with val_loss = {best_val_loss:.4f}")

print(f"\n✅ Training complete!")
print(f"   Total epochs run:      {epoch + 1}")
print(f"   Best validation loss:  {best_val_loss:.4f}")

### Visualizing the training history

Plotting the training and validation loss over epochs tells us:

- **Are we learning?** — Both curves should go down.
- **Are we overfitting?** — If validation loss starts rising while training loss keeps falling, the model is memorizing training data.
- **Did we train long enough?** — If the curves are still dropping, more epochs might help.

In [ ]:
# Plot how the training and validation loss changed over time
plot_training_history(train_losses, val_losses)

---

# 📏 5. Model Evaluation

Now for the real test: how well does the model predict prices for houses **it has never seen**?

We evaluate on the **test set** — data that was held out from both training and validation.

## Metrics we compute

**MAE (Mean Absolute Error):** — Average absolute error in dollars.

**RMSE (Root Mean Squared Error):**  — Penalizes large errors more heavily.

**MAPE (Mean Absolute Percentage Error):**  — Error as a percentage of the actual price.



In [ ]:
# Run the model on the test set and convert predictions back to real dollar values
test_predictions = get_predictions(model, test_features_t, price_scaler)

In [ ]:
# Print MAE, RMSE, and MAPE — overall and split by price range
print_performance_metrics(test_prices_original, test_predictions)

---

# 📊 6. Prediction Visualizations

Numbers alone don't always tell the full story. Let's create three complementary visualizations:

1. **Predicted vs Actual** scatter plot — ideally, all points should lie on the diagonal.
2. **Residuals histogram** — the distribution of errors; a narrow, centered bell-shape is ideal.
3. **Sample comparison** — a bar chart comparing actual and predicted prices for 20 random houses.

In [ ]:
# Show 3 charts comparing what the model predicted vs the actual prices
plot_predictions(test_prices_original, test_predictions)

---

## 🎯 Summary

In this notebook we built a **complete neural network pipeline** from scratch:

1. ✅ Loaded and explored the data
2. ✅ Preprocessed features with standardization
3. ✅ Built a 4-layer network with BatchNorm and Dropout
4. ✅ Trained with early stopping
5. ✅ Evaluated on held-out test data
6. ✅ Visualized predictions

**In the next notebook**, we'll apply some specific tricks to improve this model's performance. Stay tuned!